In [ ]:
!pip install opencv-python numpy --q

In [ ]:
"""
Red Patch Detection in Images using HSV Color Space
----------------------------------------------------
Configure the settings below, then run:
    python detect_red_patches.py
"""

import os
import sys
import zipfile
import tempfile
from pathlib import Path

import cv2
import numpy as np


# ── HSV ranges for red ────────────────────────────────────────────────────────
# Red wraps around 0°/360° in the Hue wheel, so two ranges are needed.
RED_LOWER1 = np.array([0,   120,  70])   # lower bound, range 1
RED_UPPER1 = np.array([10,  255, 255])   # upper bound, range 1
RED_LOWER2 = np.array([170, 120,  70])   # lower bound, range 2
RED_UPPER2 = np.array([180, 255, 255])   # upper bound, range 2


def detect_red_patches(image_path: str, min_area: int = 500):
    """
    Detect red patches in a single image.

    Parameters
    ----------
    image_path : str
        Path to the image file.
    min_area : int
        Minimum contour area (pixels²) to keep as a valid red patch.

    Returns
    -------
    annotated : np.ndarray
        BGR image with bounding boxes and patch count drawn on it.
    mask : np.ndarray
        Binary mask of detected red regions.
    patches : list[dict]
        List of detected patches with keys: bbox (x,y,w,h), area, centroid (cx,cy).
    """
    bgr = cv2.imread(image_path)
    if bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    # Convert to HSV
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

    # Create masks for both red hue ranges and combine
    mask1 = cv2.inRange(hsv, RED_LOWER1, RED_UPPER1)
    mask2 = cv2.inRange(hsv, RED_LOWER2, RED_UPPER2)
    mask = cv2.bitwise_or(mask1, mask2)

    # Morphological cleanup — remove noise and fill small holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    patches = []
    annotated = bgr.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        x, y, w, h = cv2.boundingRect(cnt)
        cx = x + w // 2
        cy = y + h // 2

        patches.append({"bbox": (x, y, w, h), "area": area, "centroid": (cx, cy)})

        # Draw bounding box and centroid
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(annotated,    (cx, cy),                5, (255, 0, 0), -1)
        cv2.putText(
            annotated,
            f"Red ({int(area)}px²)",
            (x, max(y - 8, 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
        )

    # Summary text
    cv2.putText(
        annotated,
        f"Red patches: {len(patches)}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 0, 255),
        2,
    )

    return annotated, mask, patches


def process_zip(zip_path: str, output_dir: str, min_area: int, save_mask: bool, show: bool):
    """Extract images from a zip archive and run red-patch detection on each."""

    SUPPORTED = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        image_entries = [
            name for name in zf.namelist()
            if Path(name).suffix.lower() in SUPPORTED and not name.startswith("__MACOSX")
        ]

        if not image_entries:
            print("No supported image files found in the zip archive.")
            return

        print(f"Found {len(image_entries)} image(s) in '{zip_path}'\n")

        with tempfile.TemporaryDirectory() as tmp_dir:
            for entry in image_entries:
                zf.extract(entry, tmp_dir)
                src = os.path.join(tmp_dir, entry)

                try:
                    annotated, mask, patches = detect_red_patches(src, min_area)
                except ValueError as exc:
                    print(f"  [SKIP] {entry}: {exc}")
                    continue

                # Build output filename (flatten nested zip paths)
                safe_name = entry.replace("/", "_").replace("\\", "_")
                stem      = Path(safe_name).stem
                suffix    = Path(safe_name).suffix

                out_path = os.path.join(output_dir, f"{stem}_detected{suffix}")
                cv2.imwrite(out_path, annotated)

                if save_mask:
                    mask_path = os.path.join(output_dir, f"{stem}_mask{suffix}")
                    cv2.imwrite(mask_path, mask)

                # Console report
                print(f"  [{entry}]")
                print(f"    Red patches found : {len(patches)}")
                for i, p in enumerate(patches, 1):
                    x, y, w, h = p["bbox"]
                    print(f"    Patch {i}: bbox=({x},{y},{w},{h})  area={int(p['area'])}px²  centroid={p['centroid']}")
                print(f"    Saved → {out_path}")
                if save_mask:
                    print(f"    Mask  → {mask_path}")
                print()

                if show:
                    cv2.imshow(f"Red Detection — {entry}", annotated)
                    cv2.imshow(f"Mask — {entry}", mask)
                    cv2.waitKey(0)
                    cv2.destroyAllWindows()

    print(f"Done. Results saved to: {os.path.abspath(output_dir)}")


def main():
    # ── Configuration — edit these values ────────────────────────────────────
    ZIP_PATH   = "/content/dataset (2).zip"              # Path to your zip file
    OUTPUT_DIR = "/content/red_detection_output"    # Folder where results will be saved
    MIN_AREA   = 500                       # Minimum patch area in pixels² (increase to ignore small specks)
    SAVE_MASK  = False                     # Set True to also save the binary HSV mask for each image
    SHOW       = False                     # Set True to display images interactively (needs a display)
    # ─────────────────────────────────────────────────────────────────────────

    if not os.path.isfile(ZIP_PATH):
        print(f"Error: '{ZIP_PATH}' not found. Update ZIP_PATH in main().")
        sys.exit(1)

    process_zip(
        zip_path   = ZIP_PATH,
        output_dir = OUTPUT_DIR,
        min_area   = MIN_AREA,
        save_mask  = SAVE_MASK,
        show       = SHOW,
    )


if __name__ == "__main__":
    main()

Found 89 image(s) in '/content/dataset (2).zip'

  [dataset/images/img_0011_full.jpg]
    Red patches found : 1
    Patch 1: bbox=(517,134,100,136)  area=13321px²  centroid=(567, 202)
    Saved → /content/red_detection_output/dataset_images_img_0011_full_detected.jpg

  [dataset/images/img_0026_full.jpg]
    Red patches found : 1
    Patch 1: bbox=(42,438,180,86)  area=15134px²  centroid=(132, 481)
    Saved → /content/red_detection_output/dataset_images_img_0026_full_detected.jpg

  [dataset/images/img_0079_partial.jpg]
    Red patches found : 1
    Patch 1: bbox=(0,469,29,120)  area=3246px²  centroid=(14, 529)
    Saved → /content/red_detection_output/dataset_images_img_0079_partial_detected.jpg

  [dataset/images/img_0025_full.jpg]
    Red patches found : 1
    Patch 1: bbox=(112,140,136,203)  area=27072px²  centroid=(180, 241)
    Saved → /content/red_detection_output/dataset_images_img_0025_full_detected.jpg

  [dataset/images/img_0064_partial.jpg]
    Red patches found : 1
    Pa

In [ ]:
"""
Red Patch Detection — Flat (Top-Down) vs Perspective (Front-Facing, 50°)
=========================================================================
Pipeline
--------
1. Load images from a zip file.
2. Run red-patch detection on the ORIGINAL flat (top-down) image.
3. Apply a perspective warp that simulates a front-facing camera tilted 50°
   downward (bird's-eye → front-view projection).
4. Run red-patch detection on the WARPED image.
5. Back-project detected centroids from the warped image to the original
   coordinate space for a fair positional comparison.
6. Estimate real-world distance to each patch centroid (front-view geometry).
7. Print a detailed accuracy/comparison report and save annotated images.

Usage
-----
Edit the CONFIGURATION block in main() then run:
    python detect_red_with_perspective.py
"""

import os
import sys
import zipfile
import tempfile
import math
from pathlib import Path

import cv2
import numpy as np


# ── HSV ranges for red ────────────────────────────────────────────────────────
RED_LOWER1 = np.array([0,   120,  70])
RED_UPPER1 = np.array([10,  255, 255])
RED_LOWER2 = np.array([170, 120,  70])
RED_UPPER2 = np.array([180, 255, 255])


# ─────────────────────────────────────────────────────────────────────────────
# Core: red-patch detector (unchanged from original)
# ─────────────────────────────────────────────────────────────────────────────

def detect_red_patches(bgr: np.ndarray, min_area: int = 500):
    """
    Detect red patches in a BGR image array.

    Returns
    -------
    annotated : np.ndarray   BGR image with bounding boxes drawn.
    mask      : np.ndarray   Binary mask of red regions.
    patches   : list[dict]   Each dict has keys: bbox, area, centroid.
    """
    hsv   = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    mask1 = cv2.inRange(hsv, RED_LOWER1, RED_UPPER1)
    mask2 = cv2.inRange(hsv, RED_LOWER2, RED_UPPER2)
    mask  = cv2.bitwise_or(mask1, mask2)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    patches   = []
    annotated = bgr.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        # ── True centroid via image moments (not bbox centre) ──────────────
        M_cnt = cv2.moments(cnt)
        if M_cnt["m00"] == 0:
            continue
        cx = int(M_cnt["m10"] / M_cnt["m00"])
        cy = int(M_cnt["m01"] / M_cnt["m00"])

        # Bounding rect kept only for label placement & back-compat metrics
        x, y, w, h = cv2.boundingRect(cnt)

        patches.append({"bbox": (x, y, w, h), "area": area, "centroid": (cx, cy),
                         "contour": cnt})

        # ── Draw the true contour outline (not a rectangle) ────────────────
        # Smooth slightly with approxPolyDP so the outline isn't jagged
        epsilon = 0.005 * cv2.arcLength(cnt, closed=True)
        approx  = cv2.approxPolyDP(cnt, epsilon, closed=True)
        cv2.drawContours(annotated, [approx], -1, (0, 255, 0), 2)

        # Centroid dot
        cv2.circle(annotated, (cx, cy), 5, (255, 0, 0), -1)

        # Label just above the topmost point of the contour
        top_y = int(cnt[:, :, 1].min())
        cv2.putText(
            annotated,
            f"Red ({int(area)}px\xb2)",
            (x, max(top_y - 8, 10)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2,
        )

    cv2.putText(
        annotated,
        f"Red patches: {len(patches)}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2,
    )
    return annotated, mask, patches


# ─────────────────────────────────────────────────────────────────────────────
# Perspective augmentation — flat → front-facing camera at `tilt_deg` degrees
# ─────────────────────────────────────────────────────────────────────────────

def build_perspective_matrix(h: int, w: int, tilt_deg: float = 50.0):
    """
    Compute a homography H that maps the flat top-down image into a perspective
    view as seen by a camera tilted `tilt_deg` degrees below horizontal
    (i.e. looking forward and downward at the ground).

    Intuition
    ---------
    A camera tilted 50° downward sees the far edge of the ground plane at a
    shallower angle than the near edge.  The transformation compresses the top
    portion of the image (far ground) more than the bottom (near ground),
    reproducing the classic "road receding to the horizon" look.

    Parameters
    ----------
    h, w       : image dimensions
    tilt_deg   : camera tilt from horizontal (0° = looking straight forward,
                 90° = looking straight down).  Typical dash-cam ≈ 50°.

    Returns
    -------
    M          : 3×3 homography (src → dst)
    M_inv      : inverse homography (dst → src), used for back-projection
    """
    tilt_rad = math.radians(tilt_deg)

    # Source corners (flat image): top-left, top-right, bottom-right, bottom-left
    src = np.float32([
        [0,     0    ],
        [w - 1, 0    ],
        [w - 1, h - 1],
        [0,     h - 1],
    ])

    # Perspective compression factor at the top of the image.
    # cos(tilt) controls how much the far edge is compressed.
    # At 50°: cos(50°) ≈ 0.643  →  top shrinks to ~64% of width.
    compress = math.cos(tilt_rad)          # lateral compression at horizon
    vanish_y = h * (1 - math.sin(tilt_rad))  # approximate vanishing-point row

    half_w   = (w - 1) / 2.0
    top_half = half_w * compress           # half-width of top edge in output

    dst = np.float32([
        [half_w - top_half, vanish_y * 0.15],   # top-left  (compressed inward)
        [half_w + top_half, vanish_y * 0.15],   # top-right (compressed inward)
        [w - 1,             h - 1           ],   # bottom-right (unchanged)
        [0,                 h - 1           ],   # bottom-left  (unchanged)
    ])

    M     = cv2.getPerspectiveTransform(src, dst)
    M_inv = cv2.getPerspectiveTransform(dst, src)
    return M, M_inv


def apply_perspective(bgr: np.ndarray, tilt_deg: float = 50.0):
    """
    Warp `bgr` to simulate a front-facing camera view at `tilt_deg`.

    Returns
    -------
    warped : np.ndarray  the warped image (same resolution)
    M      : 3×3 forward homography
    M_inv  : 3×3 inverse homography
    """
    h, w = bgr.shape[:2]
    M, M_inv = build_perspective_matrix(h, w, tilt_deg)
    warped = cv2.warpPerspective(bgr, M, (w, h), flags=cv2.INTER_LINEAR,
                                  borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    return warped, M, M_inv


# ─────────────────────────────────────────────────────────────────────────────
# Back-projection: warp-space centroid → original image space
# ─────────────────────────────────────────────────────────────────────────────

def backproject_points(points, M_inv):
    """
    Map a list of (x, y) pixel coords from the warped image back to the
    original flat image using the inverse homography.

    Parameters
    ----------
    points : list[(x, y)]
    M_inv  : 3×3 inverse homography matrix

    Returns
    -------
    list[(x_orig, y_orig)]
    """
    if not points:
        return []
    pts = np.float32(points).reshape(-1, 1, 2)
    orig = cv2.perspectiveTransform(pts, M_inv)
    return [(float(p[0][0]), float(p[0][1])) for p in orig]


# ─────────────────────────────────────────────────────────────────────────────
# Distance estimation — two models, same physical scenario
#
# Both functions return the SLANT distance (straight line, camera -> patch),
# i.e.  sqrt(ground_distance^2 + camera_height^2).
# This is the same physical quantity in both views, so the difference between
# them is the true measurement error introduced by the perspective projection.
# ─────────────────────────────────────────────────────────────────────────────

def distance_topdown(cx_pixel: float, cy_pixel: float,
                     img_width: int,  img_height: int,
                     camera_height_m: float,
                     meters_per_pixel: float):
    """
    Slant distance from the top-down (nadir) camera to a patch centroid.

    The nadir camera looks straight down.  Its optical axis hits the ground
    directly below the mounting point, which we treat as the image centre.

        dx_m = (cx - img_width/2)  * meters_per_pixel
        dy_m = (cy - img_height/2) * meters_per_pixel
        ground_dist = hypot(dx_m, dy_m)
        slant_dist  = hypot(ground_dist, camera_height)
    """
    dx_m     = (cx_pixel - img_width  / 2.0) * meters_per_pixel
    dy_m     = (cy_pixel - img_height / 2.0) * meters_per_pixel
    ground_m = math.hypot(dx_m, dy_m)
    slant_m  = math.hypot(ground_m, camera_height_m)
    return slant_m, ground_m


def distance_frontfacing(cy_pixel: float, img_height: int,
                         camera_height_m: float,
                         tilt_deg: float,
                         vfov_deg: float):
    """
    Slant distance from the front-facing (tilted) camera to a patch centroid.

        v               = cy_pixel / img_height
        angle_from_axis = (0.5 - v) * vfov_deg
        depression_deg  = tilt_deg - angle_from_axis
        ground_dist     = camera_height / tan(depression_deg)
        slant_dist      = hypot(ground_dist, camera_height)

    Returns (None, None) if the centroid is above the horizon.
    """
    v               = cy_pixel / img_height
    angle_from_axis = (0.5 - v) * vfov_deg
    depression_deg  = tilt_deg - angle_from_axis
    if depression_deg <= 0:
        return None, None
    depression_rad = math.radians(depression_deg)
    ground_m = camera_height_m / math.tan(depression_rad)
    slant_m  = math.hypot(ground_m, camera_height_m)
    return slant_m, ground_m


# kept for internal back-compat in compare_results
def estimate_distance(cy_pixel: float, img_height: int,
                      camera_height_m: float = 1.2,
                      tilt_deg: float        = 50.0,
                      vfov_deg: float        = 60.0):
    """
    Estimate the ground-plane distance (metres) to a point at pixel row `cy`
    using pinhole camera geometry.

    Model
    -----
    The camera is mounted at height `camera_height_m` above the ground,
    tilted `tilt_deg` below horizontal, with a vertical field-of-view of
    `vfov_deg` degrees.

    For a pixel at normalised vertical position  v ∈ [0, 1]  (0 = top, 1 = bottom):
        angle_from_optical_axis = (0.5 - v) * vfov_deg          (degrees)
        depression_angle        = tilt_deg - angle_from_optical_axis
        ground_distance         = camera_height / tan(depression_angle)

    Parameters
    ----------
    cy_pixel       : centroid row in the front-facing (warped) image
    img_height     : image height in pixels
    camera_height_m: camera mounting height above ground (metres)
    tilt_deg       : camera tilt from horizontal (degrees)
    vfov_deg       : vertical field of view (degrees)

    Returns
    -------
    distance_m : float  estimated ground distance in metres,
                        or None if the point is above the horizon.
    """
    v = cy_pixel / img_height                      # normalised [0, 1]
    angle_from_axis = (0.5 - v) * vfov_deg         # + above axis, - below
    depression_deg  = tilt_deg - angle_from_axis    # angle below horizontal
    if depression_deg <= 0:
        return None                                  # above horizon → undefined
    depression_rad = math.radians(depression_deg)
    return camera_height_m / math.tan(depression_rad)


# ─────────────────────────────────────────────────────────────────────────────
# Comparison / accuracy report
# ─────────────────────────────────────────────────────────────────────────────

def compare_results(flat_patches, warped_patches, M_inv,
                    img_width, img_height,
                    camera_height_m, tilt_deg, vfov_deg,
                    meters_per_pixel):
    """
    For every matched patch pair compute the camera-to-patch slant distance
    from BOTH views and report the absolute and relative distance error.

    The top-down distance is the ground truth reference because the nadir
    image has no foreshortening.  The front-facing distance is the estimate
    we are testing.  The error is:

        abs_error  = |dist_front - dist_topdown|   (metres)
        rel_error  = abs_error / dist_topdown * 100 (%)

    Both distances are SLANT distances (straight line camera -> patch),
    so they measure the same physical quantity and can be directly compared.
    """
    lines = []
    lines.append("=" * 68)
    lines.append("  DISTANCE ERROR: TOP-DOWN vs FRONT-FACING (50 deg)")
    lines.append("  Camera->patch slant distance compared for each matched patch")
    lines.append("=" * 68)
    lines.append(f"  Camera height   : {camera_height_m} m")
    lines.append(f"  Camera tilt     : {tilt_deg} deg  (front-facing view)")
    lines.append(f"  Vertical FoV    : {vfov_deg} deg")
    lines.append(f"  Scale (flat)    : {meters_per_pixel} m/px")
    lines.append(f"  Flat patches    : {len(flat_patches)}")
    lines.append(f"  Persp patches   : {len(warped_patches)}")

    # Back-project warped centroids into flat image space for matching
    warped_centroids_px = [p["centroid"] for p in warped_patches]
    bp_centroids        = backproject_points(warped_centroids_px, M_inv)
    flat_centroids      = [p["centroid"] for p in flat_patches]

    # ── Greedy nearest-neighbour matching in flat image space ─────────────────
    matched_flat   = [False] * len(flat_patches)
    matched_warped = [False] * len(warped_patches)
    pairs          = []   # (flat_idx, warped_idx)
    MATCH_THRESH   = 80   # pixels

    for wi, bp in enumerate(bp_centroids):
        best_dist, best_fi = float("inf"), -1
        for fi, fc in enumerate(flat_centroids):
            if matched_flat[fi]:
                continue
            d = math.hypot(bp[0] - fc[0], bp[1] - fc[1])
            if d < best_dist:
                best_dist, best_fi = d, fi
        if best_fi >= 0 and best_dist < MATCH_THRESH:
            matched_flat[best_fi] = True
            matched_warped[wi]    = True
            pairs.append((best_fi, wi))

    tp = len(pairs)
    fp = matched_warped.count(False)
    fn = matched_flat.count(False)

    lines.append(f"  Matched pairs   : {tp}  |  unmatched flat: {fn}  |  "
                 f"extra perspective: {fp}")
    lines.append("")

    # ── Per-pair distance comparison ──────────────────────────────────────────
    lines.append("  PER-PATCH DISTANCE COMPARISON")
    lines.append("  " + "-" * 64)
    lines.append(f"  {'Patch':<10} {'TopDown slant':>15} {'Front slant':>13} "
                 f"{'Abs error':>11} {'Rel error':>10}")
    lines.append(f"  {'':10} {'(ground truth)':>15} {'(estimated)':>13} "
                 f"{'(metres)':>11} {'(%)':>10}")
    lines.append("  " + "-" * 64)

    all_abs_errors = []
    all_rel_errors = []

    for fi, wi in pairs:
        fp_data = flat_patches[fi]
        wp_data = warped_patches[wi]

        cx_f, cy_f = fp_data["centroid"]
        cy_w       = wp_data["centroid"][1]

        # Ground-truth distance from top-down view
        slant_td, ground_td = distance_topdown(
            cx_f, cy_f, img_width, img_height,
            camera_height_m, meters_per_pixel)

        # Estimated distance from front-facing view
        slant_ff, ground_ff = distance_frontfacing(
            cy_w, img_height, camera_height_m, tilt_deg, vfov_deg)

        if slant_ff is None:
            lines.append(f"  flat#{fi+1:<5} persp#{wi+1:<4} "
                         f"{slant_td:>14.3f}m  {'above horizon':>13}  {'N/A':>11}  {'N/A':>10}")
            continue

        abs_err = abs(slant_ff - slant_td)
        rel_err = abs_err / slant_td * 100 if slant_td > 0 else float("nan")
        all_abs_errors.append(abs_err)
        all_rel_errors.append(rel_err)

        label = f"flat#{fi+1} <-> persp#{wi+1}"
        lines.append(f"  {label:<10} {slant_td:>14.3f} m  {slant_ff:>11.3f} m  "
                     f"{abs_err:>10.3f} m  {rel_err:>9.1f}%")
        lines.append(f"  {'':10}   ground(ref)={ground_td:.3f} m   "
                     f"ground(est)={ground_ff:.3f} m")
        lines.append("")

    # ── Aggregate error summary ───────────────────────────────────────────────
    lines.append("  " + "-" * 64)
    if all_abs_errors:
        lines.append("  AGGREGATE DISTANCE ERROR SUMMARY")
        lines.append(f"    Mean absolute error  : {sum(all_abs_errors)/len(all_abs_errors):.3f} m")
        lines.append(f"    Max  absolute error  : {max(all_abs_errors):.3f} m")
        lines.append(f"    Min  absolute error  : {min(all_abs_errors):.3f} m")
        lines.append(f"    Mean relative error  : {sum(all_rel_errors)/len(all_rel_errors):.1f} %")
        lines.append(f"    Max  relative error  : {max(all_rel_errors):.1f} %")
    else:
        lines.append("  No matched pairs — cannot compute distance error.")

    # ── Unmatched patches ─────────────────────────────────────────────────────
    for fi, seen in enumerate(matched_flat):
        if not seen:
            p = flat_patches[fi]
            cx, cy = p["centroid"]
            slant, _ = distance_topdown(cx, cy, img_width, img_height,
                                        camera_height_m, meters_per_pixel)
            lines.append(f"  [MISSED] flat#{fi+1}  top-down dist = {slant:.3f} m  "
                         f"(not detected in perspective view)")

    for wi, seen in enumerate(matched_warped):
        if not seen:
            p = warped_patches[wi]
            slant_ff, _ = distance_frontfacing(
                p["centroid"][1], img_height, camera_height_m, tilt_deg, vfov_deg)
            dist_str = f"{slant_ff:.3f} m" if slant_ff else "above horizon"
            lines.append(f"  [EXTRA]  persp#{wi+1}  front dist = {dist_str}  "
                         f"(no corresponding patch in flat view)")

    lines.append("")
    lines.append("  HOW TO READ THIS")
    lines.append("  ─────────────────────────────────────────────────────────────")
    lines.append("  Top-down slant  = sqrt(ground_dist^2 + camera_height^2)")
    lines.append("    where ground_dist is measured from the image centre using")
    lines.append("    meters_per_pixel.  This is the ground-truth reference.")
    lines.append("  Front-facing slant = camera_height / tan(depression_angle),")
    lines.append("    then sqrt(ground^2 + height^2).  Error comes from the")
    lines.append("    foreshortening introduced by the perspective projection.")
    lines.append("  Tip: set meters_per_pixel precisely (e.g. place a known-size")
    lines.append("    object in frame) for reliable absolute distance values.")
    lines.append("=" * 68)

    report  = "\n".join(lines)
    metrics = dict(tp=tp, fp=fp, fn=fn,
                   mean_abs_err=sum(all_abs_errors)/len(all_abs_errors) if all_abs_errors else None,
                   mean_rel_err=sum(all_rel_errors)/len(all_rel_errors) if all_rel_errors else None)
    return report, metrics


# ─────────────────────────────────────────────────────────────────────────────
# Zip processor
# ─────────────────────────────────────────────────────────────────────────────

SUPPORTED = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}


def process_zip(zip_path, output_dir, min_area,
                tilt_deg, camera_height_m, vfov_deg,
                meters_per_pixel, save_mask, show):

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        entries = [
            n for n in zf.namelist()
            if Path(n).suffix.lower() in SUPPORTED
            and not n.startswith("__MACOSX")
        ]

        if not entries:
            print("No supported images found in the archive.")
            return

        print(f"Found {len(entries)} image(s) in '{zip_path}'\n")

        with tempfile.TemporaryDirectory() as tmp:
            for entry in entries:
                zf.extract(entry, tmp)
                src = os.path.join(tmp, entry)

                bgr = cv2.imread(src)
                if bgr is None:
                    print(f"  [SKIP] {entry}: cannot read")
                    continue

                h, w = bgr.shape[:2]
                safe  = entry.replace("/", "_").replace("\\", "_")
                stem  = Path(safe).stem
                ext   = Path(safe).suffix

                # ── 1. Flat detection ─────────────────────────────────────
                ann_flat, mask_flat, patches_flat = detect_red_patches(bgr, min_area)
                cv2.imwrite(os.path.join(output_dir, f"{stem}_flat_detected{ext}"), ann_flat)
                if save_mask:
                    cv2.imwrite(os.path.join(output_dir, f"{stem}_flat_mask{ext}"), mask_flat)

                # ── 2. Perspective warp ────────────────────────────────────
                warped, M, M_inv = apply_perspective(bgr, tilt_deg)

                # Optional: draw horizon line on warped image for reference
                warped_vis = warped.copy()
                horizon_y  = int(h * (1 - math.sin(math.radians(tilt_deg))) * 0.15)
                cv2.line(warped_vis, (0, horizon_y), (w - 1, horizon_y), (0, 200, 255), 1)
                cv2.putText(warped_vis, "horizon", (5, max(horizon_y - 4, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 255), 1)

                cv2.imwrite(os.path.join(output_dir, f"{stem}_perspective_raw{ext}"), warped_vis)

                # ── 3. Perspective detection ───────────────────────────────
                ann_warp, mask_warp, patches_warp = detect_red_patches(warped, min_area)
                cv2.imwrite(os.path.join(output_dir, f"{stem}_perspective_detected{ext}"), ann_warp)
                if save_mask:
                    cv2.imwrite(os.path.join(output_dir, f"{stem}_perspective_mask{ext}"), mask_warp)

                # ── 4. Side-by-side comparison image ──────────────────────
                divider     = np.full((h, 4, 3), 200, dtype=np.uint8)
                side_by_side = np.hstack([ann_flat, divider, ann_warp])
                cv2.putText(side_by_side, "FLAT (top-down)", (10, h - 15),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                cv2.putText(side_by_side, f"PERSPECTIVE ({tilt_deg:.0f}\u00b0)",
                            (w + 14, h - 15),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                sbs_path = os.path.join(output_dir, f"{stem}_comparison{ext}")
                cv2.imwrite(sbs_path, side_by_side)

                # ── 5. Distance error report ──────────────────────────────
                report, metrics = compare_results(
                    patches_flat, patches_warp, M_inv,
                    img_width=w,
                    img_height=h,
                    camera_height_m=camera_height_m,
                    tilt_deg=tilt_deg,
                    vfov_deg=vfov_deg,
                    meters_per_pixel=meters_per_pixel,
                )

                print(f"\n{'─'*68}")
                print(f"  Image: {entry}")
                print(report)

                # Save report as text
                rpt_path = os.path.join(output_dir, f"{stem}_report.txt")
                with open(rpt_path, "w", encoding="utf-8") as f:
                    f.write(f"Image: {entry}\n\n")
                    f.write(report)
                print(f"  Report saved → {rpt_path}")

                if show:
                    cv2.imshow("Flat detection",        ann_flat)
                    cv2.imshow("Perspective (warped)",   ann_warp)
                    cv2.imshow("Side-by-side",           side_by_side)
                    cv2.waitKey(0)
                    cv2.destroyAllWindows()

    print(f"\nAll results saved to: {os.path.abspath(output_dir)}")


# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────

def main():
    # ── CONFIGURATION — edit these values ────────────────────────────────────

    ZIP_PATH   = "/content/dataset (2).zip"       # Path to your zip file
    OUTPUT_DIR = "/content/red_detection_output"  # Output folder

    # Detection
    MIN_AREA   = 10          # Minimum patch area in pixels² (ignore smaller blobs)

    # Camera geometry
    TILT_DEG         = 50.0   # Camera tilt from horizontal in degrees
    CAMERA_HEIGHT_M  = 10.0    # Camera height above ground in metres
    VFOV_DEG         = 60.0   # Vertical field-of-view of the camera in degrees
    # Scale of the flat (top-down) image.
    # How to measure: place an object of known size (e.g. 1 m ruler) in the
    # frame, count how many pixels it spans, then:
    #   METERS_PER_PIXEL = known_size_m / pixels_spanning_object
    # Default 0.005 = 5 mm/px (a ~2 m wide patch fills ~400 px at 1080p).
    METERS_PER_PIXEL = 0.005

    # Extras
    SAVE_MASK = False
    SHOW      = False

    # ─────────────────────────────────────────────────────────────────────────

    if not os.path.isfile(ZIP_PATH):
        print(f"Error: '{ZIP_PATH}' not found.  Update ZIP_PATH in main().")
        sys.exit(1)

    process_zip(
        zip_path        = ZIP_PATH,
        output_dir      = OUTPUT_DIR,
        min_area        = MIN_AREA,
        tilt_deg        = TILT_DEG,
        camera_height_m = CAMERA_HEIGHT_M,
        vfov_deg        = VFOV_DEG,
        meters_per_pixel= METERS_PER_PIXEL,
        save_mask       = SAVE_MASK,
        show            = SHOW,
    )


if __name__ == "__main__":
    main()

In [ ]:
"""
Red Patch Detection in Images using HSV Color Space
----------------------------------------------------
Detects red patches using contours (filled polygons) instead of bounding rectangles.
Usage: update IMAGE_PATH in main(), then run the script.
"""

import os
import sys
import time
from pathlib import Path

import cv2
import numpy as np


# ── HSV ranges for red ────────────────────────────────────────────────────────
RED_LOWER1 = np.array([0,   80,  40])   # range 1: pure red side
RED_UPPER1 = np.array([15,  255, 255])

RED_LOWER2 = np.array([165, 80,  40])   # range 2: wraparound red side
RED_UPPER2 = np.array([180, 255, 255])

def detect_red_patches(image_path: str, min_area: int = 500):
    """
    Detect red patches in a single image using filled contours.

    Parameters
    ----------
    image_path : str
        Path to the image file.
    min_area : int
        Minimum contour area (pixels²) to keep as a valid red patch.

    Returns
    -------
    annotated : np.ndarray
        BGR image with contours and patch count drawn on it.
    mask : np.ndarray
        Binary mask of detected red regions.
    patches : list[dict]
        List of detected patches with keys: area, centroid (cx, cy).
    elapsed : float
        Time taken to process the image in seconds.
    """
    start = time.perf_counter()

    bgr = cv2.imread(image_path)
    if bgr is None:
        raise ValueError(f"Could not read image: {image_path}")

    # Convert to HSV
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

    # Create masks for both red hue ranges and combine
    mask1 = cv2.inRange(hsv, RED_LOWER1, RED_UPPER1)
    mask2 = cv2.inRange(hsv, RED_LOWER2, RED_UPPER2)
    mask = cv2.bitwise_or(mask1, mask2)

    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    patches = []
    annotated = bgr.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        # Compute centroid via image moments
        M  = cv2.moments(cnt)
        cx = int(M["m10"] / M["m00"]) if M["m00"] != 0 else 0
        cy = int(M["m01"] / M["m00"]) if M["m00"] != 0 else 0

        patches.append({"area": area, "centroid": (cx, cy)})

        # Draw filled contour with semi-transparency
        overlay = annotated.copy()
        cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), thickness=cv2.FILLED)
        cv2.addWeighted(overlay, 0.35, annotated, 0.65, 0, annotated)

        # Draw contour outline
        cv2.drawContours(annotated, [cnt], -1, (0, 255, 0), 2)

        # Draw centroid dot
        cv2.circle(annotated, (cx, cy), 5, (255, 0, 0), -1)

        # Label near centroid
        cv2.putText(
            annotated,
            f"Red ({int(area)}px\u00b2)",
            (cx + 8, cy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
        )

    elapsed = time.perf_counter() - start

    # Summary text
    cv2.putText(
        annotated,
        f"Red patches: {len(patches)}  |  {elapsed*1000:.1f} ms",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 0, 255),
        2,
    )

    return annotated, mask, patches, elapsed


def process_image(image_path: str, output_dir: str, min_area: int, save_mask: bool, show: bool):
    """Run red-patch detection on a single image file."""

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    if not os.path.isfile(image_path):
        print(f"Error: '{image_path}' not found.")
        sys.exit(1)

    try:
        annotated, mask, patches, elapsed = detect_red_patches(image_path, min_area)
    except ValueError as exc:
        print(f"[SKIP] {exc}")
        sys.exit(1)

    stem   = Path(image_path).stem
    suffix = Path(image_path).suffix or ".jpg"

    out_path = os.path.join(output_dir, f"{stem}_detected{suffix}")
    cv2.imwrite(out_path, annotated)

    mask_path = None
    if save_mask:
        mask_path = os.path.join(output_dir, f"{stem}_mask{suffix}")
        cv2.imwrite(mask_path, mask)

    # Console report
    print(f"Image       : {image_path}")
    print(f"Time taken  : {elapsed*1000:.2f} ms")
    print(f"Red patches : {len(patches)}")
    for i, p in enumerate(patches, 1):
        print(f"  Patch {i}: area={int(p['area'])}px²  centroid={p['centroid']}")
    print(f"Saved       → {os.path.abspath(out_path)}")
    if mask_path:
        print(f"Mask        → {os.path.abspath(mask_path)}")

    if show:
        cv2.imshow("Red Detection", annotated)
        cv2.imshow("Mask", mask)
        cv2.waitKey(0)
        cv2.destroyAllWindows()


def main():
    # ── Configuration — edit these values ────────────────────────────────────
    IMAGE_PATH = "/content/30474361-cadd-486e-a2a9-11305e55810a.jpg"        # Path to your image file
    OUTPUT_DIR = "/content/red_detection_output"  # Folder where results will be saved
    MIN_AREA   = 100                              # Minimum patch area in pixels²
    SAVE_MASK  = False                            # Set True to also save the binary HSV mask
    SHOW       = False                            # Set True to display images interactively
    # ─────────────────────────────────────────────────────────────────────────

    process_image(
        image_path = IMAGE_PATH,
        output_dir = OUTPUT_DIR,
        min_area   = MIN_AREA,
        save_mask  = SAVE_MASK,
        show       = SHOW,
    )


if __name__ == "__main__":
    main()

Image       : /content/30474361-cadd-486e-a2a9-11305e55810a.jpg
Time taken  : 26.94 ms
Red patches : 1
  Patch 1: area=6131px²  centroid=(575, 249)
Saved       → /content/red_detection_output/30474361-cadd-486e-a2a9-11305e55810a_detected.jpg
